In [ ]:
# ======================
# IMPORTS
# ======================
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing import image

print("TensorFlow:", tf.__version__)

# ======================
# CONFIG
# ======================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "dataset")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Dataset path:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))

# ======================
# DATA GENERATOR
# ======================
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("Classes:", train_gen.class_indices)

# ======================
# SAVE LABELS
# ======================
labels_path = os.path.join(MODEL_DIR, "papaya_labels.json")

class_labels = {v: k for k, v in train_gen.class_indices.items()}

with open(labels_path, "w") as f:
    json.dump(class_labels, f, indent=4)

# ======================
# MODEL
# ======================
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ======================
# CALLBACKS
# ======================
checkpoint_path = os.path.join(MODEL_DIR, "papaya_model.keras")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ======================
# TRAINING
# ======================
history = model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen,
    callbacks=callbacks_list
)

# ======================
# FINE TUNING
# ======================
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen,
    callbacks=callbacks_list
)

print("Training complete!")

# ======================
# PREDICTION FUNCTION
# ======================
def predict_disease(img_path):
    model = tf.keras.models.load_model(checkpoint_path)

    with open(labels_path, "r") as f:
        labels = json.load(f)

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]
    class_idx = np.argmax(preds)
    confidence = float(preds[class_idx])

    return {
        "disease": labels[str(class_idx)],
        "confidence": round(confidence * 100, 2)
    }

# Example:
# print(predict_disease("test.jpg"))

TensorFlow: 2.21.0
Dataset path: F:\krishi nova project ml part\papaya\dataset
Exists: True
Found 14504 images belonging to 8 classes.
Found 3626 images belonging to 8 classes.
Classes: {'Anthracnose': 0, 'BacterialSpot': 1, 'Curl': 2, 'Healthy': 3, 'Mealybug': 4, 'Mite disease': 5, 'Mosaic': 6, 'Ringspot': 7}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 8)                   │           2,056 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,384,683 (16.73 MB)

 Trainable params: 332,552 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/3
454/454 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.1608 - loss: 2.3829
Epoch 1: val_accuracy improved from None to 0.21456, saving model to F:\krishi nova project ml part\papaya\model\papaya_model.keras

Epoch 1: finished saving model to F:\krishi nova project ml part\papaya\model\papaya_model.keras
454/454 ━━━━━━━━━━━━━━━━━━━━ 2713s 6s/step - accuracy: 0.1736 - loss: 2.2441 - val_accuracy: 0.2146 - val_loss: 2.0678
Epoch 2/3
454/454 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1964 - loss: 2.0355
Epoch 2: val_accuracy did not improve from 0.21456
454/454 ━━━━━━━━━━━━━━━━━━━━ 2509s 6s/step - accuracy: 0.1993 - loss: 2.0167 - val_accuracy: 0.2146 - val_loss: 2.0475
Epoch 3/3
211/454 ━━━━━━━━━━━━━━━━━━━━ 23:52 6s/step - accuracy: 0.2029 - loss: 1.9914